# Etapa 7. Produtividade atingível e partição do yield gap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/07_yield_gap.ipynb)

Com a PP verificada (Etapa 5) e o ky calibrado (Etapa 6), calculam-se a PA de todos os ciclos e as parcelas do
yield gap:

```
YG_total    = PP_janela − PR
YG_hídrico  = PP_janela − PA_janela   (falta de água na janela em que o produtor semeia), dividido em:
    YG_data       = PA_melhor data − PA_janela      (parte que a melhor data de semeadura evitaria)
    YG_inevitável = YG_hídrico − YG_data            (parte que nenhuma data evita)
YG_manejo   = PA_janela − PR                        (todo o resto: adubação, híbrido, pragas, colheita...)
```

Assim, YG_total = YG_inevitável + YG_data + YG_manejo. A **melhor data** de cada município é a de maior PA média
ao longo dos anos. Como a melhor data pode ter PP maior que a da janela, o ganho com a data é limitado ao próprio
YG_hídrico, para que nenhuma parcela fique negativa.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1 pymannkendall
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mza_oeste as M

sim = pd.read_parquet(SAIDA / "simulacao_mza_oeste.parquet")
ibge = pd.read_csv("dados/ibge_pam_milho_oeste.csv")
ibge = ibge[(ibge["safra"] == 2) & (ibge["area_colhida_ha"] >= 100)]

FATOR_PP = json.load(open(SAIDA / "parametros.json"))["FATOR_PP"]
KY_CAL = json.load(open(SAIDA / "ky_calibrado.json"))["ky"]
print("Fator da PP:", FATOR_PP)
print("ky calibrado:", KY_CAL)

## 1. PP e PA de cada ciclo, e o nível da PA

A PA tem de ficar **acima** da produtividade real na grande maioria dos casos, porque o produtor não colhe mais do
que a água permite. Se a PR passar da PA com frequência, o yield gap de manejo sai **negativo**, o que não tem
sentido físico: é sinal de que o balanço hídrico superestima o déficit (clima da sede do município, raiz rasa, Kc).

Há duas formas de calcular a PA, e a célula abaixo compara as duas:

- **absoluto:** PA = PP × EC, com a EC direta do modelo;
- **normalizado:** PA = PP × EC / EC₉₀, em que EC₉₀ é o quantil 0,9 da EC do município. É a **mesma escala usada
  na calibração** (Etapa 6): admite que, nos anos bons, a água não limitou a produtividade.

A escolha é uma **decisão de método**, a ser discutida com o orientador e justificada no artigo.

In [ ]:
sim["PP"] = sim["PPf"] * FATOR_PP
ehjanela = sim["semeadura"].isin(M.JANELA)

def pa(ky, nivel):
    """PA de cada ciclo com o ky dado, no nível "absoluto" ou "normalizado"."""
    ec = M.ec(sim, ky)
    if nivel == "absoluto":
        return sim["PP"] * ec
    ec_janela = sim.assign(ec=ec)[ehjanela].groupby(["cod_ibge", "ano"])["ec"].mean()
    ec90 = ec_janela.groupby("cod_ibge").quantile(0.9)                 # EC dos anos bons de cada município
    return sim["PP"] * (ec / sim["cod_ibge"].map(ec90)).clip(upper=1.0)

diag = {}
for nivel in ["absoluto", "normalizado"]:
    j = (sim.assign(PA=pa(KY_CAL, nivel))[ehjanela].groupby(["cod_ibge", "ano"])[["PP", "PA"]].mean()
         .reset_index().merge(ibge, on=["cod_ibge", "ano"]))
    diag[nivel] = {"PP média": j["PP"].mean(), "PA média": j["PA"].mean(), "PR média": j["rendimento_kg_ha"].mean(),
                   "% município-anos com PR > PA": 100 * (j["rendimento_kg_ha"] > j["PA"]).mean()}
pd.DataFrame(diag).round(1)

In [ ]:
# >>> DECISÃO (com o orientador) <<<
NIVEL = "normalizado"

sim["PA_cal"] = pa(KY_CAL, NIVEL)          # PA com o ky calibrado
sim["PA_dk"] = pa(M.KY_DK, NIVEL)          # PA com o ky tabelado, para comparar

# melhor data de cada município: a de maior PA média nos 27 anos
media_data = sim.groupby(["cod_ibge", "semeadura"])["PA_cal"].mean()
melhor = media_data.groupby("cod_ibge").idxmax().str[1].rename("melhor_data")
print(melhor.value_counts())

In [ ]:
janela = (sim[sim["semeadura"].isin(M.JANELA)]
          .groupby(["cod_ibge", "municipio", "ano"])[["PP", "PA_cal", "PA_dk"]].mean())
na_melhor = (sim[sim["semeadura"] == sim["cod_ibge"].map(melhor)]
             .set_index(["cod_ibge", "ano"])["PA_cal"].rename("PA_melhor"))

gap = (janela.reset_index()
       .merge(na_melhor.reset_index(), on=["cod_ibge", "ano"])
       .merge(ibge[["cod_ibge", "ano", "rendimento_kg_ha", "area_colhida_ha"]], on=["cod_ibge", "ano"])
       .rename(columns={"rendimento_kg_ha": "PR"}))
gap["YG_total"] = gap["PP"] - gap["PR"]
hidrico = gap["PP"] - gap["PA_cal"]                                     # gap hídrico na janela do produtor
gap["YG_data"] = (gap["PA_melhor"] - gap["PA_cal"]).clip(lower=0).clip(upper=hidrico)
gap["YG_hidrico"] = hidrico - gap["YG_data"]                            # parte inevitável
gap["YG_manejo"] = gap["PA_cal"] - gap["PR"]
gap["YG_explotavel"] = 0.8 * gap["PA_cal"] - gap["PR"]       # só ~80% da PA é alcançável na prática
gap.to_parquet(SAIDA / "yield_gap.parquet", index=False)
print(f"Município-anos: {len(gap)};  YG_manejo negativo em {(gap['YG_manejo'] < 0).mean():.1%} (PR acima da PA)")

## 2. Quanto é de cada

Médias da região, em kg/ha e em porcentagem do YG_total. A média ponderada pela área colhida dá mais peso aos
municípios que mais produzem.

In [ ]:
parcelas = ["YG_hidrico", "YG_data", "YG_manejo"]
w = gap["area_colhida_ha"]
resumo = pd.DataFrame({
    "kg/ha (média simples)": gap[["PP", "PA_melhor", "PA_cal", "PR", "YG_total"] + parcelas].mean(),
    "kg/ha (ponderada)": gap[["PP", "PA_melhor", "PA_cal", "PR", "YG_total"] + parcelas].mul(w, axis=0).sum() / w.sum(),
})
resumo["% do YG_total"] = 100 * resumo["kg/ha (ponderada)"] / resumo.loc["YG_total", "kg/ha (ponderada)"]
resumo.loc[["PP", "PA_melhor", "PA_cal", "PR"], "% do YG_total"] = np.nan
resumo.round(1)

## 3. Ano a ano

In [ ]:
por_ano = gap.groupby("ano")[parcelas + ["PR", "YG_total"]].mean()
fig, ax = plt.subplots(figsize=(11, 5))
base = np.zeros(len(por_ano))
cores = {"YG_hidrico": "#d7301f", "YG_data": "#fc8d59", "YG_manejo": "#4575b4"}
rotulos = {"YG_hidrico": "Hídrico inevitável", "YG_data": "Data de semeadura", "YG_manejo": "Manejo"}
ax.bar(por_ano.index, por_ano["PR"], color="#9e9e9e", label="Produtividade real (IBGE)")
base = por_ano["PR"].to_numpy().copy()
for p in parcelas:
    ax.bar(por_ano.index, por_ano[p], bottom=base, color=cores[p], label=rotulos[p])
    base = base + por_ano[p].to_numpy()
ax.set_ylabel("kg/ha"); ax.legend(ncol=2); ax.set_title("A soma das barras é a PP da janela")
fig.savefig(SAIDA / "fig_particao_ano.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. O gap de manejo está fechando?

O teste de **Mann-Kendall** diz se há tendência monotônica na série (p < 0,05 = tendência significativa), e a
**inclinação de Sen** dá o tamanho da tendência (kg/ha por ano). São testes não paramétricos, adequados a séries
curtas e sem distribuição normal.

In [ ]:
import pymannkendall as mk

for p in ["YG_manejo", "YG_hidrico", "YG_total"]:
    t = mk.original_test(por_ano[p])
    print(f"{p:11s} tendência: {t.trend:15s} p = {t.p:.3f}   Sen = {t.slope:6.1f} kg/ha por ano")

## 5. Mapas

Cada município colorido pela média de uma parcela. O `geopandas` lê os limites municipais e faz mapas como se fosse
uma tabela.

In [ ]:
import geopandas as gpd

malha = gpd.read_file("dados/malha_municipios_oeste.geojson")
media_mun = gap.groupby("cod_ibge")[parcelas + ["YG_total"]].mean()
mapa = malha.merge(media_mun, left_on="cod_ibge", right_index=True)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, p in zip(axs, parcelas):
    mapa.plot(column=p, cmap="YlOrRd", legend=True, edgecolor="grey", lw=0.3, ax=ax,
              legend_kwds={"label": "kg/ha", "shrink": 0.7})
    ax.set_title(rotulos[p]); ax.set_axis_off()
fig.savefig(SAIDA / "fig_mapas_yield_gap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Quanto o ky calibrado muda a resposta

Mesma partição em duas parcelas (hídrico = PP − PA; manejo = PA − PR) com o ky tabelado e com o calibrado. Este
número mostra por que calibrar importa.

In [ ]:
comp = pd.DataFrame({
    "ky tabelado": {"hídrico (%)": 100 * (gap["PP"] - gap["PA_dk"]).mean() / gap["YG_total"].mean(),
                    "manejo (%)": 100 * (gap["PA_dk"] - gap["PR"]).mean() / gap["YG_total"].mean()},
    "ky calibrado": {"hídrico (%)": 100 * (gap["PP"] - gap["PA_cal"]).mean() / gap["YG_total"].mean(),
                     "manejo (%)": 100 * (gap["PA_cal"] - gap["PR"]).mean() / gap["YG_total"].mean()},
})
comp.round(1)

### Exercício e extensão

1. Quais anos tiveram o maior YG_hídrico? Batem com os anos de quebra do notebook 02?
2. Os municípios com maior YG_manejo têm algo em comum (área colhida pequena, solo, localização)?
3. **Extensão (opcional):** baixe o índice ONI do ENSO (NOAA) e compare o YG_hídrico médio em anos de La Niña,
   El Niño e neutros.

### O que escrever

O núcleo de **Resultados e Discussão**: a tabela da partição, a figura por ano, os mapas e a tendência do gap de
manejo. Em cada parágrafo: primeiro o número, depois a explicação agronômica, por fim a comparação com outros
autores (Sentelhas et al., 2015; Andrea et al., 2018; Soares et al., 2025).